# Trabajo Práctico N°2 — Preprocesamiento

**Alumno:** Gonzalo Zarazaga

---

Este notebook aplica las decisiones de preprocesamiento definidas durante el EDA (`02_eda.ipynb`).
El mismo pipeline se aplica tanto al dataset de entrenamiento como al dataset de predicción,
garantizando que el modelo vea exactamente el mismo formato de datos en ambos contextos.

**Regla fundamental:** todo lo que se haga sobre los datos de entrenamiento debe replicarse
exactamente sobre los datos de predicción, usando los parámetros derivados del conjunto de entrenamiento.

**Insumos:**
- `data/raw/smoking_prediction.csv` — conjunto de entrenamiento (50.000 filas, etiquetado)
- `data/raw/smoking_prediction_entrega.csv` — conjunto de predicción (5.692 filas, sin etiquetar)

**Salidas:**
- `data/processed/smoking_train_processed.csv` — datos de entrenamiento listos para modelar
- `data/processed/smoking_predict_processed.csv` — datos de predicción listos para inferencia
- `models/categorical_mappings.json` — mapeos de codificación para reproducibilidad

In [1]:
import pandas as pd
import numpy as np
import json
from pathlib import Path

PATH_TRAIN   = "../data/raw/smoking_prediction.csv"
PATH_PREDICT = "../data/raw/smoking_prediction_entrega.csv"
PATH_PROC    = Path("../data/processed")
PATH_MODELS  = Path("../models")

PATH_PROC.mkdir(exist_ok=True)
PATH_MODELS.mkdir(exist_ok=True)

print("Carpetas listas:")
print(f"  {PATH_PROC.resolve()}")
print(f"  {PATH_MODELS.resolve()}")

Carpetas listas:
  /Users/gonzalo/workspace/Diplomatura/TP2/data/processed
  /Users/gonzalo/workspace/Diplomatura/TP2/models


## 1. Carga de los datasets

Se cargan ambos datasets en su estado original (sin ninguna transformación previa)
y se verifica que las columnas sean las esperadas.

In [2]:
df_train   = pd.read_csv(PATH_TRAIN)
df_predict = pd.read_csv(PATH_PREDICT)

print(f"Dataset de entrenamiento: {df_train.shape[0]:,} filas × {df_train.shape[1]} columnas")
print(f"Dataset de predicción:    {df_predict.shape[0]:,} filas × {df_predict.shape[1]} columnas")

Dataset de entrenamiento: 50,000 filas × 27 columnas
Dataset de predicción:    5,692 filas × 26 columnas


In [3]:
# Verificar diferencias de columnas entre ambos datasets
cols_train   = set(df_train.columns)
cols_predict = set(df_predict.columns)

solo_en_train   = cols_train   - cols_predict
solo_en_predict = cols_predict - cols_train

print(f"Solo en entrenamiento (columna a predecir): {solo_en_train}")
print(f"Solo en predicción:                         {solo_en_predict}")
print(f"Columnas comunes: {len(cols_train & cols_predict)}")

# Confirmar que no hay nulos en ninguno
print(f"\nNulos en train:   {df_train.isna().sum().sum()}")
print(f"Nulos en predict: {df_predict.isna().sum().sum()}")

Solo en entrenamiento (columna a predecir): {'smoking'}
Solo en predicción:                         set()
Columnas comunes: 26

Nulos en train:   0
Nulos en predict: 0


## 2. Estrategia de preprocesamiento

Basado en los hallazgos del EDA, se aplican las siguientes transformaciones:

| Acción | Variables | Motivo |
|---|---|---|
| **Eliminar** | `ID` (solo train) | Identificador técnico, no es una feature |
| **Eliminar** | `oral` (ambos) | Variable constante: único valor `"Y"` → sin varianza, sin poder predictivo |
| **Codificar** | `gender` | Categórica binaria → `M=1, F=0` |
| **Codificar** | `tartar` | Categórica binaria → `Y=1, N=0` |
| **No imputar** | todas | Sin valores nulos en ninguno de los dos datasets |
| **No tratar outliers** | numéricas | XGBoost es robusto a valores atípicos |
| **No escalar** | numéricas | XGBoost no requiere normalización |

> El dataset de predicción conserva `ID` para poder asociar cada predicción con su registro original.

## 3. Eliminación de columnas irrelevantes

In [4]:
COLS_ELIMINAR = ["oral"]  # constante en ambos datasets

# Entrenamiento: se elimina ID (no se necesita para modelar) y oral
df_train_proc = df_train.drop(columns=["ID"] + COLS_ELIMINAR)

# Predicción: se conserva ID para la entrega final, solo se elimina oral
df_predict_proc = df_predict.drop(columns=COLS_ELIMINAR)

print(f"Entrenamiento — antes: {df_train.shape}  →  después: {df_train_proc.shape}")
print(f"Predicción    — antes: {df_predict.shape}  →  después: {df_predict_proc.shape}")

print(f"\nColumnas eliminadas de entrenamiento: {['ID'] + COLS_ELIMINAR}")
print(f"Columnas eliminadas de predicción:    {COLS_ELIMINAR}")

Entrenamiento — antes: (50000, 27)  →  después: (50000, 25)
Predicción    — antes: (5692, 26)  →  después: (5692, 25)

Columnas eliminadas de entrenamiento: ['ID', 'oral']
Columnas eliminadas de predicción:    ['oral']


## 4. Codificación de variables categóricas

Las variables `gender` y `tartar` son categóricas binarias. Se convierten a valores numéricos
usando mapeos deterministas. El mapeo se define a partir del conjunto de entrenamiento
y se aplica de forma idéntica al conjunto de predicción.

| Variable | Original | Codificado |
|---|---|---|
| `gender` | `F` | `0` |
| `gender` | `M` | `1` |
| `tartar` | `N` | `0` |
| `tartar` | `Y` | `1` |

In [5]:
# Mapeos definidos sobre los valores observados en el conjunto de entrenamiento
MAPEOS_CATEGORICOS = {
    "gender": {"F": 0, "M": 1},
    "tartar": {"N": 0, "Y": 1}
}

for col, mapeo in MAPEOS_CATEGORICOS.items():
    df_train_proc[col]   = df_train_proc[col].map(mapeo)
    df_predict_proc[col] = df_predict_proc[col].map(mapeo)

# Verificación: no deben quedar valores nulos (indicaría un valor no visto en entrenamiento)
print("Verificación post-codificación:")
for col in MAPEOS_CATEGORICOS:
    nulos_train   = df_train_proc[col].isna().sum()
    nulos_predict = df_predict_proc[col].isna().sum()
    vals_train    = sorted(df_train_proc[col].unique())
    vals_predict  = sorted(df_predict_proc[col].unique())
    print(f"  '{col}' — train: {vals_train} (nulos: {nulos_train}) | predict: {vals_predict} (nulos: {nulos_predict})")

Verificación post-codificación:
  'gender' — train: [np.int64(0), np.int64(1)] (nulos: 0) | predict: [np.int64(0), np.int64(1)] (nulos: 0)
  'tartar' — train: [np.int64(0), np.int64(1)] (nulos: 0) | predict: [np.int64(0), np.int64(1)] (nulos: 0)


## 5. Verificación del dataset procesado

In [6]:
print("=== Dataset de entrenamiento procesado ===")
df_train_proc.info()
print(f"\nNulos totales: {df_train_proc.isna().sum().sum()}")
print(f"Columnas tipo object: {list(df_train_proc.select_dtypes('object').columns)} (debe ser vacío)")

=== Dataset de entrenamiento procesado ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 25 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   gender               50000 non-null  int64  
 1   age                  50000 non-null  int64  
 2   height(cm)           50000 non-null  int64  
 3   weight(kg)           50000 non-null  int64  
 4   waist(cm)            50000 non-null  float64
 5   eyesight(left)       50000 non-null  float64
 6   eyesight(right)      50000 non-null  float64
 7   hearing(left)        50000 non-null  float64
 8   hearing(right)       50000 non-null  float64
 9   systolic             50000 non-null  float64
 10  relaxation           50000 non-null  float64
 11  fasting blood sugar  50000 non-null  float64
 12  Cholesterol          50000 non-null  float64
 13  triglyceride         50000 non-null  float64
 14  HDL                  50000 non-null  float6

In [7]:
df_train_proc.head(5)

,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),systolic,...,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,tartar,smoking
0,0,40,155,60,3.38,0.04,0.04,0.04,0.04,4.75,...,5.25,0.51,0.04,0.00,0.75,0.79,1.13,0,1,0
1,0,40,160,60,3.38,0.01,0.00,0.04,0.04,4.96,...,5.29,0.50,0.04,0.00,0.92,0.79,0.75,0,1,0
2,1,55,170,60,3.33,0.01,0.01,0.04,0.04,5.75,...,6.29,0.63,0.04,0.04,0.88,0.67,0.92,0,0,1
3,1,40,165,70,3.67,0.05,0.05,0.04,0.04,4.17,...,9.42,0.59,0.04,0.04,0.79,1.08,0.75,0,1,0
4,0,40,155,60,3.58,0.04,0.04,0.04,0.04,5.00,...,4.46,0.50,0.04,0.00,0.67,0.58,0.92,0,0,0


In [8]:
print("=== Dataset de predicción procesado ===")
print(f"Shape: {df_predict_proc.shape}")
print(f"Nulos totales: {df_predict_proc.isna().sum().sum()}")
df_predict_proc.head(5)

=== Dataset de predicción procesado ===
Shape: (5692, 25)
Nulos totales: 0


,ID,gender,age,height(cm),weight(kg),waist(cm),eyesight(left),eyesight(right),hearing(left),hearing(right),...,HDL,LDL,hemoglobin,Urine protein,serum creatinine,AST,ALT,Gtp,dental caries,tartar
0,27358,1,25,160,65,3.42,0.05,0.00,0.04,0.04,...,1.96,3.04,0.63,0.04,0.01,0.75,0.71,0.71,0,1
1,27364,1,30,180,80,3.46,0.04,0.01,0.04,0.04,...,1.50,4.21,0.59,0.04,0.04,0.79,1.13,1.33,0,0
2,27368,1,55,165,60,3.42,0.00,0.01,0.04,0.04,...,2.13,2.04,0.63,0.04,0.01,1.08,1.29,2.00,1,1
3,27378,1,20,175,75,3.63,0.05,0.05,0.04,0.04,...,1.71,3.71,0.63,0.04,0.04,0.83,0.58,0.46,0,0
4,27381,1,25,165,80,3.79,0.04,0.04,0.04,0.04,...,2.38,6.63,0.67,0.04,0.04,1.25,1.63,1.96,1,1


## 6. Separación de features y target

Se separan las variables predictoras (`X`) de la variable objetivo (`y`) en el conjunto de entrenamiento.
XGBoost requiere que `y` sea numérico — la columna `smoking` ya es `0`/`1`.

In [9]:
TARGET = "smoking"

X_train = df_train_proc.drop(columns=[TARGET])
y_train = df_train_proc[TARGET]

print(f"X_train: {X_train.shape}  ← features")
print(f"y_train: {y_train.shape}  ← target")
print(f"\ny_train dtype: {y_train.dtype}  (XGBoost requiere numérico: OK)")
print(f"\nDistribución del target:")
print(y_train.value_counts().rename({0: 'No fumador (0)', 1: 'Fumador (1)'}).to_string())
print(f"\nColumnas de features ({len(X_train.columns)}):")
for col in X_train.columns:
    print(f"  {col:<30} {X_train[col].dtype}")

X_train: (50000, 24)  ← features
y_train: (50000,)  ← target

y_train dtype: int64  (XGBoost requiere numérico: OK)

Distribución del target:
smoking
No fumador (0)    31671
Fumador (1)       18329

Columnas de features (24):
  gender                         int64
  age                            int64
  height(cm)                     int64
  weight(kg)                     int64
  waist(cm)                      float64
  eyesight(left)                 float64
  eyesight(right)                float64
  hearing(left)                  float64
  hearing(right)                 float64
  systolic                       float64
  relaxation                     float64
  fasting blood sugar            float64
  Cholesterol                    float64
  triglyceride                   float64
  HDL                            float64
  LDL                            float64
  hemoglobin                     float64
  Urine protein                  float64
  serum creatinine               float64
  A

## 7. Exportación de archivos procesados

Se exportan los datasets procesados a `data/processed/` y los artefactos del pipeline a `models/`.

| Archivo | Contenido |
|---|---|
| `data/processed/smoking_train_processed.csv` | Features + target (`smoking`), sin `ID` ni `oral` |
| `data/processed/smoking_predict_processed.csv` | Features + `ID`, sin `oral` ni `smoking` |
| `models/categorical_mappings.json` | Mapeos de codificación para reproducir el pipeline |

In [10]:
# Exportar datasets procesados
out_train   = PATH_PROC / "smoking_train_processed.csv"
out_predict = PATH_PROC / "smoking_predict_processed.csv"

df_train_proc.to_csv(out_train, index=False)
df_predict_proc.to_csv(out_predict, index=False)

print(f"✓ {out_train}")
print(f"  → {df_train_proc.shape[0]:,} filas × {df_train_proc.shape[1]} columnas")
print()
print(f"✓ {out_predict}")
print(f"  → {df_predict_proc.shape[0]:,} filas × {df_predict_proc.shape[1]} columnas")

✓ ../data/processed/smoking_train_processed.csv
  → 50,000 filas × 25 columnas

✓ ../data/processed/smoking_predict_processed.csv
  → 5,692 filas × 25 columnas


In [11]:
# Exportar mapeos de codificación
out_mappings = PATH_MODELS / "categorical_mappings.json"

with open(out_mappings, "w", encoding="utf-8") as f:
    json.dump(MAPEOS_CATEGORICOS, f, indent=2, ensure_ascii=False)

print(f"✓ {out_mappings}")
print()
print("Contenido del archivo:")
print(json.dumps(MAPEOS_CATEGORICOS, indent=2))
print()
print("Cómo usarlo en otros notebooks:")
print("  import json")
print("  with open('../models/categorical_mappings.json') as f:")
print("      mapeos = json.load(f)")
print("  for col, m in mapeos.items():")
print("      df[col] = df[col].map(m)")

✓ ../models/categorical_mappings.json

Contenido del archivo:
{
  "gender": {
    "F": 0,
    "M": 1
  },
  "tartar": {
    "N": 0,
    "Y": 1
  }
}

Cómo usarlo en otros notebooks:
  import json
  with open('../models/categorical_mappings.json') as f:
      mapeos = json.load(f)
  for col, m in mapeos.items():
      df[col] = df[col].map(m)


## 8. Resumen del preprocesamiento

### Transformaciones aplicadas

| Paso | Acción | Train | Predict |
|---|---|---|---|
| Eliminar `ID` | Drop | ✓ | ✗ (conservado para entrega) |
| Eliminar `oral` | Drop | ✓ | ✓ |
| Codificar `gender` | M→1, F→0 | ✓ | ✓ |
| Codificar `tartar` | Y→1, N→0 | ✓ | ✓ |
| Imputación de nulos | — | No aplica | No aplica |
| Tratamiento de outliers | — | No aplica | No aplica |
| Escalado de variables | — | No aplica | No aplica |

### Resultado

| Dataset | Filas | Columnas | Tipo |
|---|---|---|---|
| `smoking_train_processed.csv` | 50.000 | 25 | Features + target |
| `smoking_predict_processed.csv` | 5.692 | 25 | ID + features |

### Próximo paso

El notebook `04_entrenamiento_y_optimizacion.ipynb` carga `smoking_train_processed.csv`,
divide en train/test con estratificación y entrena el modelo XGBoost.